# RBY1 Real Robot Inference with LAP (Right Arm Only)

GR00T 실로봇 inference notebook 구조를 기반으로, **LAP 체크포인트 + WebSocket 정책 서버**를 사용하도록 구현한 버전입니다.

## 핵심 차이점 (vs GR00T)

| 항목 | GR00T | LAP |
|------|-------|-----|
| Policy | ZMQ 서버 (`Gr00tPolicy.get_action()`) | WebSocket 서버 (`WebsocketClientPolicy.infer()`) |
| Action 차원 | 16D (bimanual) | **8D** (right arm only: `right_arm[7] + right_gripper[1]`) |
| Action 형식 | dict of arrays (부위별 분리) | `actions` array `(horizon, 8)` — **이미 absolute joint target** |
| Left Arm | 모델이 직접 제어 | **현재 위치 유지** (hold) |
| 이미지 | 3캠 (head + left_wrist + right_wrist) | **2캠** (head + right_wrist) |
| State 입력 | 16D | 16D (서버 pipeline이 내부적으로 8D로 slice) |
| Normalization | Processor 내부 | 서버 pipeline (Normalize → Unnormalize) |

## 서버 실행 방법 (별도 터미널)

```bash
cd /lustre/meat124/lap
uv run scripts/serve_policy.py \
  --policy.config lap_rby1 \
  --policy.dir checkpoints/lap_rby1/<exp_name>/<step>
```

## 연결 구성

| 대상 | 주소 | 비고 |
|------|------|------|
| 실로봇 | `localhost:50051` | 직접 연결 |
| 시뮬레이터 | `localhost:50052` | Docker 포트 매핑 |

### 시뮬레이터 실행 (별도 터미널)
```bash
sudo docker run --rm -it \
  -e DISPLAY=$DISPLAY \
  -v /tmp/.X11-unix:/tmp/.X11-unix \
  -v /exe \
  -p 50052:50051 \
  rainbowroboticsofficial/rby1-sim
```

## 실행 순서
1. 설정/연결 정보 로드 + LAP Policy 서버 연결
2. 초기 자세(initial pose) 이동 — 실로봇 (safe path + head)
3. Gripper 초기화 (homing + 열기)
4. RealSense 카메라 + standalone observation 구성
5. Dry-run inference (로봇 명령 없이 출력만 확인)
6. [선택] 시뮬레이터 초기 자세 이동 + action chunk 테스트
7. [선택] 시뮬레이터 Episode 루프 (inference → action chunk 전송 반복)
8. 실로봇 Episode Loop (inference → right arm action chunk 전송)
9. 시각화 + 정리

## 주의
- LAP 정책 서버를 먼저 실행한 뒤 진행하세요.
- 시뮬레이터를 사용하려면 Docker를 미리 실행하세요.
- 실로봇 실행 전 주변 안전을 반드시 확보하세요.

## Step 1 — 라이브러리 Import 및 기본 설정

필요한 라이브러리를 불러오고, 로봇 IP, 정책 서버 주소, 카메라 시리얼, 제어 파라미터 등 전역 설정값을 정의합니다.

> **항상 가장 먼저 실행해야 하는 셀입니다.**

In [ ]:
from __future__ import annotations

import gc
import logging
import sys
import time
from pathlib import Path

import cv2
import numpy as np
import pyrealsense2 as rs
import rby1_sdk as rby

# scripts/deployment/ 을 sys.path에 추가 — rby1_remote_gripper 등의 모듈 임포트용
_deployment_dir = str(Path("/home/hyunjin/rby1_ws/lap/scripts/deployment"))
if _deployment_dir not in sys.path:
    sys.path.insert(0, _deployment_dir)

logging.basicConfig(level=logging.INFO, force=True)

# ---------- User Config ----------
ROBOT_IP = "localhost:50051"
SIM_IP = "localhost:50052"

# LAP policy server (WebSocket)
POLICY_SERVER_HOST = "0.0.0.0"
POLICY_SERVER_PORT = 8000

PROMPT = "pick up the grey cup and place it on the plate"

# RealSense serials (head + right wrist only for LAP)
CAM_HEAD_SERIAL = "838212070714"
CAM_RIGHT_SERIAL = "838212074317"

# LAP model parameters (overridden by server metadata if available)
ACTION_DIM = 8        # right_arm[7] + right_gripper[1]
ACTION_HORIZON = 16

# Test mode — if True, simulator/single-chunk cells run
TEST_MODE = False

# Runtime/Control
EPISODE_LENGTH = 400

# ------ RTC (Real-Time Chunking) ------
# 서버를 --use-rtc 플래그로 시작했을 때 아래 USE_RTC = True 로 설정하세요.
# 서버 시작 예시 (--policy-config / --policy-dir 사용, subcommand 없음):
#   cd /home/hyunjin/rby1_ws/lap
#   uv run scripts/serve_policy.py \
#       --policy-config lap_rby1_eef \
#       --policy-dir checkpoints/lap_rby1_eef/PuttingCupintotheDish_demo50_uniform_eef/40000 \
#       --use-rtc --rtc-execution-horizon 8 --rtc-max-guidance-weight 1.0 --rtc-schedule linear
USE_RTC                = True
RTC_MAX_GUIDANCE_WEIGHT = 1.0
RTC_EXECUTION_HORIZON  = 8
RTC_SCHEDULE           = "linear"   # "linear" | "exp" | "ones" | "zeros"

EXECUTE_CHUNK_SIZE = RTC_EXECUTION_HORIZON if USE_RTC else None  # None이면 predict chunk 전체 실행

ARM_COMMAND_PRIORITY = 10
ARM_MINIMUM_TIME = 10.0
USE_REMOTE_GRIPPER = True

# Initial pose (safe path + head init)
ENABLE_INITIAL_POSE = True
SAFE_INIT_PATH = True
INIT_COMMAND_PRIORITY = 10
INIT_DT = 0.05
INIT_HOLD_TIME = 0.2
ENABLE_HEAD_INIT = True
HEAD_INIT_DEG = np.array([0.0, 40.0], dtype=np.float64)
HEAD_INIT = np.deg2rad(HEAD_INIT_DEG)

# rby1 joint layout (24-DOF + grippers)
TORSO_SLICE = slice(2, 8)    # 6 joints
RIGHT_SLICE = slice(8, 15)   # 7 joints
LEFT_SLICE = slice(15, 22)   # 7 joints

print("Config loaded")
print(f"ROBOT_IP={ROBOT_IP}  SIM_IP={SIM_IP}")
print(f"POLICY_SERVER={POLICY_SERVER_HOST}:{POLICY_SERVER_PORT}")
print(f"CAMERAS head={CAM_HEAD_SERIAL} right_wrist={CAM_RIGHT_SERIAL}")
print(f"ACTION_DIM={ACTION_DIM}  ACTION_HORIZON={ACTION_HORIZON}")
print(f"PROMPT={PROMPT}")
print(f"USE_RTC={USE_RTC}  EXECUTE_CHUNK_SIZE={EXECUTE_CHUNK_SIZE}")
print(f"deployment dir on sys.path: {_deployment_dir}")


## Step 1-2 — LAP Policy 서버 연결 + Helper 함수 정의

`openpi_client.WebsocketClientPolicy`로 LAP 정책 서버에 연결하고, observation 변환 / action 변환 helper 함수를 정의합니다.

### 모드 자동 감지
서버 메타데이터의 `"eef": True` 여부로 **Joint 모드 / EEF 모드**를 자동 감지합니다.

| 항목 | Joint 모드 (`lap_rby1`) | EEF 모드 (`lap_rby1_eef`) |
|------|-------------------------|---------------------------|
| `ACTION_DIM` | 8 | 7 |
| State 입력 | 8D joint `[right_arm(7), gripper(1)]` | 7D EEF `[x,y,z,r,p,y,gripper]` (FK 계산) |
| Action 형식 | absolute joint target (rad) | absolute EEF `[x,y,z,r,p,y,gripper]` |
| 로봇 명령 | `JointPositionCommandBuilder` | `CartesianCommandBuilder` (SDK IK) |

### Helper 함수
- `build_lap_input()`: observation → LAP server input (모드별 자동 분기)
- `parse_lap_action()`: server response → action array
- `compute_eef_from_joints()`: (EEF) joint → FK → 7D EEF pose
- `apply_eef_delta()`: (EEF) current EEF + delta → target EEF
- `eef_7d_to_se3()`: (EEF) 7D → 4×4 SE3 matrix
- `build_cartesian_right_arm_cmd()`: (EEF) SE3 target → robot command
- `build_joint_right_arm_cmd()`: (Joint) absolute joint target → robot command

In [ ]:
# ---------- Step 1-2: LAP Policy 서버 연결 + 모드 감지 ----------
from openpi_client import websocket_client_policy as _ws_client
from scipy.spatial.transform import Rotation as _Rotation

policy = _ws_client.WebsocketClientPolicy(
    host=POLICY_SERVER_HOST,
    port=POLICY_SERVER_PORT,
)

server_meta = policy.get_server_metadata()
ACTION_DIM = int(server_meta.get("action_dim", ACTION_DIM))
ACTION_HORIZON = int(server_meta.get("action_horizon", ACTION_HORIZON))
IS_EEF_MODE = bool(server_meta.get("eef", False))

# RTC mismatch check
server_rtc = bool(server_meta.get("rtc_enabled", False))
if USE_RTC != server_rtc:
    print(f"[WARN] RTC mismatch: notebook USE_RTC={USE_RTC} vs server rtc_enabled={server_rtc}")
    print(f"       → 서버를 --use-rtc 플래그와 함께 재시작하거나 USE_RTC={server_rtc}로 설정하세요.")

print("LAP policy server connected.")
print(f"Server metadata: {server_meta}")
print(f"ACTION_DIM={ACTION_DIM}  ACTION_HORIZON={ACTION_HORIZON}")
print(f"IS_EEF_MODE={IS_EEF_MODE}")
print(f"RTC: notebook USE_RTC={USE_RTC}, server rtc_enabled={server_rtc}")
if IS_EEF_MODE:
    print("  → EEF 모드: absolute EEF actions [x,y,z,r,p,y,gripper] + CartesianCommand")
else:
    print("  → Joint 모드: absolute joint targets [right_arm(7), gripper(1)] + JointPositionCommand")


# =====================================================================
# Forward Kinematics (EEF 모드용)
# — lap/scripts/convert_rby1_joint_to_eef.py 에서 포팅
# =====================================================================

# Kinematic chain: base → torso(6) → right_arm(7) → tool → FT_sensor → ee_right
_FK_CHAIN = [
    (np.array([0.0, 0.0, 0.2805]),           np.array([1.0, 0.0, 0.0])),  # torso_0
    (np.array([0.0, 0.0, 0.0]),              np.array([0.0, 1.0, 0.0])),  # torso_1
    (np.array([0.0, 0.0, 0.350]),            np.array([0.0, 1.0, 0.0])),  # torso_2
    (np.array([0.0, 0.0, 0.350]),            np.array([0.0, 1.0, 0.0])),  # torso_3
    (np.array([0.0, 0.0, 0.0]),              np.array([1.0, 0.0, 0.0])),  # torso_4
    (np.array([0.0, 0.0, 0.309426548461]),   np.array([0.0, 0.0, 1.0])),  # torso_5
    (np.array([0.0, -0.220, 0.080073451539]),np.array([0.0, 0.93969262, -0.34202014])),  # right_arm_0
    (np.array([0.0, 0.0, 0.0]),              np.array([1.0, 0.0, 0.0])),  # right_arm_1
    (np.array([0.0, 0.0, 0.0]),              np.array([0.0, 0.0, 1.0])),  # right_arm_2
    (np.array([0.031, 0.0, -0.276]),         np.array([0.0, 1.0, 0.0])),  # right_arm_3
    (np.array([-0.031, 0.0, -0.256]),        np.array([0.0, 0.0, 1.0])),  # right_arm_4
    (np.array([0.0, 0.0, 0.0]),              np.array([0.0, 1.0, 0.0])),  # right_arm_5
    (np.array([0.0, 0.0, 0.0]),              np.array([0.0, 0.0, 1.0])),  # right_arm_6
]
_FK_TOOL_OFFSET = np.array([0.0, 0.0, -0.1087])
_FK_FT_OFFSET = np.array([0.0, 0.0, -0.0461])

# 데이터 수집 시 고정 torso 각도 (학습 데이터와 일치)
_TORSO_FIXED_DEG = np.array([0.0, 20.0, -40.0, 35.0, 0.0, 0.0])
_TORSO_FIXED_RAD = np.deg2rad(_TORSO_FIXED_DEG)


def _rodrigues(axis: np.ndarray, angle: float) -> np.ndarray:
    """Rodrigues' rotation formula → 3×3 rotation matrix."""
    a = axis / np.linalg.norm(axis)
    K = np.array([[0, -a[2], a[1]], [a[2], 0, -a[0]], [-a[1], a[0], 0]])
    return np.eye(3) + np.sin(angle) * K + (1 - np.cos(angle)) * (K @ K)


def _hom(R: np.ndarray, t: np.ndarray) -> np.ndarray:
    """Build 4×4 homogeneous transform."""
    T = np.eye(4)
    T[:3, :3] = R
    T[:3, 3] = t
    return T


def _fk_base_to_ee(joint_angles_13: np.ndarray) -> np.ndarray:
    """base → ee_right 4×4 transform. Input: [torso(6), right_arm(7)]."""
    T = np.eye(4)
    for i, (xyz, axis) in enumerate(_FK_CHAIN):
        T = T @ _hom(np.eye(3), xyz) @ _hom(_rodrigues(axis, joint_angles_13[i]), np.zeros(3))
    T = T @ _hom(np.eye(3), _FK_TOOL_OFFSET) @ _hom(np.eye(3), _FK_FT_OFFSET)
    return T


def compute_eef_from_joints(right_arm_7: np.ndarray, gripper: float) -> np.ndarray:
    """Right arm 7D joints + gripper → 7D EEF [x,y,z,roll,pitch,yaw,gripper].

    Uses fixed torso angles from data collection for FK computation.
    """
    chain_q = np.concatenate([_TORSO_FIXED_RAD, np.asarray(right_arm_7, dtype=np.float64)])
    T = _fk_base_to_ee(chain_q)
    pos = T[:3, 3]
    euler = _Rotation.from_matrix(T[:3, :3]).as_euler("xyz", degrees=False)
    return np.concatenate([pos, euler, [gripper]]).astype(np.float32)


def apply_eef_delta(current_eef_7d: np.ndarray, delta_7d: np.ndarray) -> np.ndarray:
    """Apply EEF delta to current EEF pose → new absolute EEF pose.

    Args:
        current_eef_7d: [x,y,z,roll,pitch,yaw,gripper]
        delta_7d: [dx,dy,dz,droll,dpitch,dyaw,gripper]

    Returns:
        target_eef_7d: [x,y,z,roll,pitch,yaw,gripper]
        - Position: simple addition
        - Rotation: composition via Rotation (avoids gimbal lock)
        - Gripper: use model output directly (absolute normalized 0-1)
    """
    new_pos = current_eef_7d[:3] + delta_7d[:3]
    R_cur = _Rotation.from_euler("xyz", current_eef_7d[3:6])
    R_delta = _Rotation.from_euler("xyz", delta_7d[3:6])
    R_new = R_cur * R_delta
    new_euler = R_new.as_euler("xyz", degrees=False)
    new_gripper = delta_7d[6]  # gripper은 모델 출력 그대로 (absolute)
    return np.concatenate([new_pos, new_euler, [new_gripper]]).astype(np.float32)


def eef_7d_to_se3(eef_7d: np.ndarray) -> np.ndarray:
    """7D EEF [x,y,z,roll,pitch,yaw,...] → 4×4 SE3 matrix."""
    R = _Rotation.from_euler("xyz", eef_7d[3:6]).as_matrix()
    return _hom(R, eef_7d[:3])


# =====================================================================
# Robot command builders
# =====================================================================

def build_joint_right_arm_cmd(
    torso_hold: np.ndarray,
    right_target: np.ndarray,
    left_hold: np.ndarray,
    dt: float,
    hold_time: float,
):
    """Build joint-space right arm command (torso/left hold, right from model)."""
    return rby.RobotCommandBuilder().set_command(
        rby.ComponentBasedCommandBuilder().set_body_command(
            rby.BodyComponentBasedCommandBuilder()
            .set_torso_command(
                rby.JointPositionCommandBuilder()
                .set_command_header(rby.CommandHeaderBuilder().set_control_hold_time(hold_time))
                .set_position(torso_hold.astype(np.float64))
                .set_minimum_time(dt)
            )
            .set_right_arm_command(
                rby.JointPositionCommandBuilder()
                .set_command_header(rby.CommandHeaderBuilder().set_control_hold_time(hold_time))
                .set_position(right_target.astype(np.float64))
                .set_minimum_time(dt)
            )
            .set_left_arm_command(
                rby.JointPositionCommandBuilder()
                .set_command_header(rby.CommandHeaderBuilder().set_control_hold_time(hold_time))
                .set_position(left_hold.astype(np.float64))
                .set_minimum_time(dt)
            )
        )
    )


def build_cartesian_right_arm_cmd(
    torso_hold: np.ndarray,
    T_target: np.ndarray,
    left_hold: np.ndarray,
    hold_time: float,
    translation_weight: float = 1.0,
    rotation_weight: float = 1.0,
    velocity_scaling: float = 0.8,
):
    """Build Cartesian right arm command using SDK IK.

    Args:
        torso_hold: (6,) torso joint positions to hold
        T_target: (4,4) SE3 target for ee_right in base frame
        left_hold: (7,) left arm joint positions to hold
        hold_time: control hold time
        translation_weight: IK translation priority
        rotation_weight: IK rotation priority
        velocity_scaling: velocity limit scaling for IK
    """
    return rby.RobotCommandBuilder().set_command(
        rby.ComponentBasedCommandBuilder().set_body_command(
            rby.BodyComponentBasedCommandBuilder()
            .set_torso_command(
                rby.JointPositionCommandBuilder()
                .set_command_header(rby.CommandHeaderBuilder().set_control_hold_time(hold_time))
                .set_position(torso_hold.astype(np.float64))
                .set_minimum_time(0.1)
            )
            .set_right_arm_command(
                rby.CartesianCommandBuilder()
                .set_command_header(rby.CommandHeaderBuilder().set_control_hold_time(hold_time))
                .add_target("base", "ee_right", T_target.astype(np.float64),
                            translation_weight, rotation_weight, velocity_scaling)
            )
            .set_left_arm_command(
                rby.JointPositionCommandBuilder()
                .set_command_header(rby.CommandHeaderBuilder().set_control_hold_time(hold_time))
                .set_position(left_hold.astype(np.float64))
                .set_minimum_time(0.1)
            )
        )
    )


# =====================================================================
# Observation / Action helpers (Joint + EEF 모드 공용)
# =====================================================================

def _chw_to_hwc_uint8(image_chw: np.ndarray) -> np.ndarray:
    if image_chw.ndim == 3 and image_chw.shape[0] in (1, 3):
        image_hwc = np.transpose(image_chw, (1, 2, 0))
    else:
        image_hwc = image_chw
    if image_hwc.dtype != np.uint8:
        image_hwc = image_hwc.astype(np.uint8)
    return image_hwc

# Joint 모드: right_arm_only 슬라이스
_RIGHT_ARM_ONLY_IDX = np.array([0, 1, 2, 3, 4, 5, 6, 14], dtype=int)


def build_lap_input(
    state16: np.ndarray,
    head_image_hwc: np.ndarray,
    right_wrist_image_hwc: np.ndarray,
    prompt: str,
) -> dict:
    """Build observation dict for LAP policy server.

    Auto-branches on IS_EEF_MODE:
      - Joint mode: slice 16D → 8D [right_arm(7), right_gripper(1)]
      - EEF mode: FK → 7D [x,y,z,roll,pitch,yaw,gripper]
    """
    state16 = np.asarray(state16, dtype=np.float32)
    if IS_EEF_MODE:
        right_arm = state16[:7]
        right_grip = float(state16[14])
        state = compute_eef_from_joints(right_arm, right_grip)
    else:
        state = state16[_RIGHT_ARM_ONLY_IDX]
    return {
        "observation": {
            "base_0_rgb": head_image_hwc,
            "left_wrist_0_rgb": right_wrist_image_hwc,
            "state": state,
        },
        "prompt": prompt,
    }


def parse_lap_action(result: dict) -> np.ndarray:
    """Parse LAP server response to action array.

    Returns:
        (action_horizon, ACTION_DIM) float32 array.
        - Joint mode: absolute joint targets [right_arm(7), gripper(1)]
        - EEF mode: absolute EEF [x,y,z,r,p,y,gripper]
    """
    actions = np.asarray(result["actions"], dtype=np.float32)
    if actions.ndim == 1:
        actions = actions.reshape(1, -1)
    return actions[:, :ACTION_DIM]


print("Helper functions defined.")
if IS_EEF_MODE:
    print("  compute_eef_from_joints() : joint7 + gripper → 7D EEF (FK)")
    print("  apply_eef_delta()         : current_eef + delta → target_eef")
    print("  eef_7d_to_se3()           : 7D EEF → 4×4 SE3")
    print("  build_cartesian_right_arm_cmd() : SE3 target → CartesianCommand")
    print("  build_lap_input()         : state16 → FK → 7D EEF state → server input")
    print("  parse_lap_action()        : server response → (T, 7) absolute EEF array")
else:
    print("  build_joint_right_arm_cmd() : joint target → JointPositionCommand")
    print("  build_lap_input()           : state16 → 8D slice → server input")
    print("  parse_lap_action()          : server response → (T, 8) absolute target array")

## Step 2 — 실로봇 초기 자세 이동

로봇 팔을 안전한 waypoint 경로를 거쳐 데이터 수집 시작 자세로 이동시킵니다.

- **elbow 굽힘 여부**에 따라 짧은 경로(midpoint2 → initial) 또는 전체 safe path(midpoint1 → midpoint2 → initial)를 선택합니다.
- Head 초기화가 활성화되면 지정 각도로 이동합니다.
- Tool flange 12V를 공급하여 그리퍼 전원을 켭니다.

In [ ]:
# ---------- Step 2: initial pose 이동 (safe path + head) ----------
if ENABLE_INITIAL_POSE:
    TORSO_INIT_DEG = np.array([0.0, 20.0, -40.0, 35.0, 0.0, 0.0], dtype=np.float64)
    INIT_RIGHT_DEG = np.array([-24.0, -60.0, 10.0, -120.0, -60.0, 85.0, 0.0], dtype=np.float64)
    INIT_LEFT_DEG = np.array([-24.0, 60.0, -10.0, -120.0, 60.0, 85.0, 0.0], dtype=np.float64)
    TORSO_INIT = np.deg2rad(TORSO_INIT_DEG)
    INIT_RIGHT = np.deg2rad(INIT_RIGHT_DEG)
    INIT_LEFT = np.deg2rad(INIT_LEFT_DEG)

    MID1_RIGHT_DEG = np.array([70.0, -30.0, 0.0, -100.0, 0.0, -70.0, 0.0], dtype=np.float64)
    MID1_LEFT_DEG = np.array([70.0, 30.0, 0.0, -100.0, 0.0, -70.0, 0.0], dtype=np.float64)
    MID2_RIGHT_DEG = np.array([0.0, -15.0, 0.0, -100.0, 0.0, -70.0, 0.0], dtype=np.float64)
    MID2_LEFT_DEG = np.array([0.0, 15.0, 0.0, -100.0, 0.0, -70.0, 0.0], dtype=np.float64)
    MID1_RIGHT = np.deg2rad(MID1_RIGHT_DEG)
    MID1_LEFT = np.deg2rad(MID1_LEFT_DEG)
    MID2_RIGHT = np.deg2rad(MID2_RIGHT_DEG)
    MID2_LEFT = np.deg2rad(MID2_LEFT_DEG)

    ELBOW_INIT_THRESHOLD_DEG = 90.0

    def _build_body_head_cmd(torso_q, right_q, left_q, head_q, dt):
        body_builder = rby.BodyComponentBasedCommandBuilder()
        if torso_q is not None:
            body_builder = body_builder.set_torso_command(
                rby.JointPositionCommandBuilder()
                .set_command_header(rby.CommandHeaderBuilder().set_control_hold_time(INIT_HOLD_TIME))
                .set_position(np.asarray(torso_q, dtype=np.float64))
                .set_minimum_time(float(dt))
            )
        body_builder = (
            body_builder
            .set_right_arm_command(
                rby.JointPositionCommandBuilder()
                .set_command_header(rby.CommandHeaderBuilder().set_control_hold_time(INIT_HOLD_TIME))
                .set_position(np.asarray(right_q, dtype=np.float64))
                .set_minimum_time(float(dt))
            )
            .set_left_arm_command(
                rby.JointPositionCommandBuilder()
                .set_command_header(rby.CommandHeaderBuilder().set_control_hold_time(INIT_HOLD_TIME))
                .set_position(np.asarray(left_q, dtype=np.float64))
                .set_minimum_time(float(dt))
            )
        )
        cbc = rby.ComponentBasedCommandBuilder().set_body_command(body_builder)
        if head_q is not None:
            cbc = cbc.set_head_command(
                rby.JointPositionCommandBuilder()
                .set_command_header(rby.CommandHeaderBuilder().set_control_hold_time(INIT_HOLD_TIME))
                .set_position(np.asarray(head_q, dtype=np.float64))
                .set_minimum_time(float(dt))
            )
        return rby.RobotCommandBuilder().set_command(cbc)

    def _move_waypoint_stream(robot, stream, name, torso_target, right_target, left_target, head_target, duration):
        q0 = np.asarray(robot.get_state().position, dtype=np.float64)
        start_torso = q0[TORSO_SLICE].copy()
        start_right = q0[RIGHT_SLICE].copy()
        start_left = q0[LEFT_SLICE].copy()
        goal_torso = torso_target if torso_target is not None else start_torso
        steps = max(2, int(np.ceil(duration / INIT_DT)))
        torso_path = np.linspace(start_torso, goal_torso, num=steps)
        right_path = np.linspace(start_right, right_target, num=steps)
        left_path = np.linspace(start_left, left_target, num=steps)
        for i in range(steps):
            cmd = _build_body_head_cmd(torso_path[i], right_path[i], left_path[i], head_target, dt=INIT_DT)
            try:
                stream.send_command(cmd)
            except RuntimeError as exc:
                if "expired" in str(exc).lower():
                    robot.wait_for_control_ready(1000)
                    stream = robot.create_command_stream(priority=INIT_COMMAND_PRIORITY)
                    stream.send_command(cmd)
                else:
                    raise
            time.sleep(INIT_DT)
        q_now = np.asarray(robot.get_state().position, dtype=np.float64)
        print(f"[init-pose] reached {name} | torso_now[1]={q_now[3]:+.4f}, right_now[0]={q_now[8]:+.4f}, left_now[0]={q_now[15]:+.4f}")
        return stream

    robot_init = rby.create_robot(ROBOT_IP, "a") if hasattr(rby, "create_robot") else rby.create_robot_a(ROBOT_IP)
    robot_init.connect()
    assert robot_init.is_connected(), f"Failed to connect robot at {ROBOT_IP}"

    robot_init.power_on(".*")
    robot_init.servo_on(".*")
    robot_init.enable_control_manager()

    try:
        robot_init.cancel_control()
    except Exception:
        pass
    robot_init.wait_for_control_ready(1000)

    q0 = np.asarray(robot_init.get_state().position, dtype=np.float64)
    print("[init-pose] current torso:", np.round(q0[TORSO_SLICE], 3))
    print("[init-pose] current right:", np.round(q0[RIGHT_SLICE], 3))
    print("[init-pose] current left :", np.round(q0[LEFT_SLICE], 3))
    if ENABLE_HEAD_INIT:
        print("[init-pose] target head(deg):", np.round(HEAD_INIT_DEG, 2))

    right_elbow_now = float(q0[RIGHT_SLICE][3])
    left_elbow_now = float(q0[LEFT_SLICE][3])
    elbow_bent = (
        abs(right_elbow_now) > np.deg2rad(ELBOW_INIT_THRESHOLD_DEG)
        or abs(left_elbow_now) > np.deg2rad(ELBOW_INIT_THRESHOLD_DEG)
    )
    print(
        f"[init-pose] elbow check | right={np.degrees(right_elbow_now):+.1f}, "
        f"left={np.degrees(left_elbow_now):+.1f}  "
        f"-> {'near-INITIAL (bent)' if elbow_bent else 'near-ZERO (straight)'}"
    )

    if SAFE_INIT_PATH:
        if elbow_bent:
            waypoints = [
                ("midpoint2", None,       MID2_RIGHT, MID2_LEFT, 5.0),
                ("initial",   TORSO_INIT, INIT_RIGHT, INIT_LEFT, 3.0),
            ]
            print("[init-pose] near-initial detected: midpoint2 -> initial")
        else:
            waypoints = [
                ("midpoint1", None,       MID1_RIGHT, MID1_LEFT, 5.0),
                ("midpoint2", None,       MID2_RIGHT, MID2_LEFT, 5.0),
                ("initial",   TORSO_INIT, INIT_RIGHT, INIT_LEFT, 3.0),
            ]
            print("[init-pose] near-zero detected: midpoint1 -> midpoint2 -> initial")
    else:
        waypoints = [("initial", TORSO_INIT, INIT_RIGHT, INIT_LEFT, 4.0)]
        print("[init-pose] SAFE_INIT_PATH disabled: direct -> initial")

    head_target = HEAD_INIT if ENABLE_HEAD_INIT else None
    stream = robot_init.create_command_stream(priority=INIT_COMMAND_PRIORITY)
    for name, torso_target, right_target, left_target, duration in waypoints:
        print(f"[init-pose] move -> {name} (t={duration:.1f}s)")
        stream = _move_waypoint_stream(
            robot_init, stream, name=name,
            torso_target=torso_target, right_target=right_target,
            left_target=left_target, head_target=head_target, duration=duration,
        )

    time.sleep(0.2)
    q_end = np.asarray(robot_init.get_state().position, dtype=np.float64)
    err_t = float(np.linalg.norm(q_end[TORSO_SLICE] - TORSO_INIT))
    err_r = float(np.linalg.norm(q_end[RIGHT_SLICE] - INIT_RIGHT))
    err_l = float(np.linalg.norm(q_end[LEFT_SLICE] - INIT_LEFT))
    print(f"[init-pose] done | final error norm torso={err_t:.6f}, right={err_r:.6f}, left={err_l:.6f}")

    if USE_REMOTE_GRIPPER:
        for _arm in ("right", "left"):
            ok_v = robot_init.set_tool_flange_output_voltage(_arm, 12)
            print(f"[init-pose] tool flange 12V {_arm}: {'OK' if ok_v else 'FAILED'}")
        time.sleep(0.5)

    try:
        robot_init.cancel_control()
    except Exception:
        pass
    if hasattr(robot_init, "disconnect"):
        robot_init.disconnect()

    INITIAL_POSE_DONE = True
else:
    INITIAL_POSE_DONE = False
    print("[init-pose] skipped")

## Step 2-5 — Gripper 초기화 (Homing + 열기)

RemoteGripper UDP 클라이언트를 통해 그리퍼를 초기화합니다.

1. `initialize()` — 서버 ping 확인
2. `homing()` — 범위 캘리브레이션 (min_q / max_q)
3. `start()` — 제어 루프 시작
4. 완전 열기 (normalized=1.0)

In [ ]:
# ---------- Step 2-5: Gripper 초기화 (homing + 열기) ----------
if not globals().get("INITIAL_POSE_DONE", False):
    raise RuntimeError("먼저 Step 2 (initial pose) 셀을 실행하세요.")
if not globals().get("USE_REMOTE_GRIPPER", False):
    print("[gripper-init] USE_REMOTE_GRIPPER=False -> skip")
else:
    from rby1_remote_gripper import Gripper as RemoteGripper

    if "stream" in globals() and stream is not None:
        try:
            del stream
        except Exception:
            pass
        stream = None
        print("[gripper-init] 이전 stream 객체 정리 완료")

    if "gripper_obj" in globals() and gripper_obj is not None:
        try:
            gripper_obj.stop()
            print("[gripper-init] 이전 gripper 객체 stop() 완료")
        except Exception as _e:
            print(f"[gripper-init] 이전 gripper stop 무시: {_e}")
        gripper_obj = None

    print("[gripper-init] RemoteGripper 연결 중...")
    gripper_obj = RemoteGripper()
    print(f"[gripper-init] host={gripper_obj.host}  port={gripper_obj.port}")

    if gripper_obj.host is None or gripper_obj.port is None:
        raise RuntimeError(
            "[gripper-init] gripper host/port가 설정되지 않았습니다.\n"
            "  scripts/deployment/config.yaml 또는 환경변수 확인"
        )

    # 1) ping / initialize
    print("[gripper-init] ping 확인 중...")
    ok_init = gripper_obj.initialize(verbose=True)
    print(f"[gripper-init] ping 결과: {'OK' if ok_init else 'FAILED'}")
    if not ok_init:
        raise RuntimeError("[gripper-init] 그리퍼 서버 응답 없음.")

    # 2) homing
    print("[gripper-init] homing 중... (30초 이내)")
    ok_homing = gripper_obj.homing()
    if not ok_homing:
        raise RuntimeError("[gripper-init] homing 실패")

    _min_q = getattr(gripper_obj, "min_q", None)
    _max_q = getattr(gripper_obj, "max_q", None)
    if _min_q is None or _max_q is None:
        raise RuntimeError("[gripper-init] homing 성공했지만 min_q/max_q가 없음.")
    print(f"[gripper-init] homing 완료 | min_q={_min_q}  max_q={_max_q}")

    # 3) 제어 루프 시작
    print("[gripper-init] start() 호출 중...")
    gripper_obj.start()
    print("[gripper-init] start() 완료")

    # 4) 완전 열기
    _old_timeout = getattr(gripper_obj, "timeout", None)
    _fast_timeout = 3.0
    try:
        _old_timeout_f = float(_old_timeout) if _old_timeout is not None else None
    except Exception:
        _old_timeout_f = None
    if _old_timeout_f is None or _old_timeout_f > _fast_timeout:
        gripper_obj.timeout = _fast_timeout
        try:
            gripper_obj._sock.settimeout(_fast_timeout)
        except Exception:
            pass
    print(f"[gripper-init] fast timeout 적용: {_old_timeout} -> {gripper_obj.timeout}")

    # LAP는 오른쪽 그리퍼만 사용하지만, 양쪽 모두 열어둠
    print("[gripper-init] 완전 열기 명령 전송 (normalized=1.0)...")
    gripper_obj.set_normalized_target(np.array([1.0, 1.0]), wait_for_reply=True)
    time.sleep(0.5)

    try:
        _grip_state = gripper_obj.get_state()
        print(f"[gripper-init] 현재 그리퍼 상태: {_grip_state}")
    except Exception as _se:
        print(f"[gripper-init][WARN] 상태 조회 실패: {_se}")

    GRIPPER_INIT_DONE = True
    print("[gripper-init] 완료")

## Step 3 — RealSense 카메라 초기화 + 관측(Observation) 구성

`pyrealsense2`를 이용해 Head / Right Wrist 2대의 카메라를 초기화하고, `rby1_sdk`를 통해 로봇 state를 읽어 observation 함수를 정의합니다.

> GR00T는 `RBY1Environment` 래퍼를 사용하지만, LAP은 독립 구현합니다.
> LAP은 카메라 2대 (head + right_wrist)만 사용합니다.

In [ ]:
# ---------- Step 3: 카메라 초기화 + observation 함수 ----------
import matplotlib.pyplot as plt

CAM_SERIALS = {
    "head": CAM_HEAD_SERIAL,
    "right_wrist": CAM_RIGHT_SERIAL,
}

IMG_WIDTH = 640
IMG_HEIGHT = 480
RENDER_SIZE = 224  # LAP expects 224x224


# ---------- resize_with_pad (letterbox) ----------
def resize_with_pad(image: np.ndarray, target_h: int, target_w: int) -> np.ndarray:
    """Resize image to target size while preserving aspect ratio, padding with black."""
    in_h, in_w = image.shape[:2]
    scale = min(target_h / in_h, target_w / in_w)
    new_h = max(1, int(round(in_h * scale)))
    new_w = max(1, int(round(in_w * scale)))
    resized = cv2.resize(image, (new_w, new_h), interpolation=cv2.INTER_AREA)
    out = np.zeros((target_h, target_w, image.shape[2]), dtype=resized.dtype)
    y0 = (target_h - new_h) // 2
    x0 = (target_w - new_w) // 2
    out[y0 : y0 + new_h, x0 : x0 + new_w] = resized
    return out


# (A) RealSense 장치 진단
# query_devices()가 반환하는 일부 device는 UVC 노드 없이 유령 열거될 수 있어
# get_info() 호출 시 UVCIOC_CTRL_QUERY 에러가 발생함 → 개별 try/except로 건너뜀
_ctx = rs.context()
_available = {}
_ghost_count = 0
for _dev in _ctx.query_devices():
    try:
        _sn = _dev.get_info(rs.camera_info.serial_number)
        _nm = _dev.get_info(rs.camera_info.name)
        _available[_sn] = _nm
    except Exception as _e:
        _ghost_count += 1
        print(f"  [WARN] USB 유령 디바이스 건너뜀 ({_ghost_count}번째): {_e}")
del _ctx

print("-" * 60)
print(f"  [camera-check] 감지된 정상 디바이스: {len(_available)}  유령: {_ghost_count}")
for serial, name in _available.items():
    cfg_name = next((n for n, s in CAM_SERIALS.items() if s == serial), "unknown")
    mark = "OK" if serial in CAM_SERIALS.values() else "INFO"
    print(f"  {mark:4s} {cfg_name:>12} | {name} | S/N: {serial}")

_not_found = [(n, s) for n, s in CAM_SERIALS.items() if s not in _available]
if _not_found:
    for n, s in _not_found:
        print(f"  MISSING {n}: S/N {s}")
    print()
    print("  [힌트] USB 유령 디바이스가 있는 경우 다음 명령으로 uvcvideo 모듈을 리로드하세요:")
    print("    sudo modprobe -r uvcvideo && sudo modprobe uvcvideo")
    print("  또는 해당 카메라 USB 케이블을 물리적으로 재연결해 주세요.")
    raise RuntimeError(
        f"설정된 카메라를 찾을 수 없습니다: {[n for n, _ in _not_found]}\n"
        "USB 연결 상태를 확인하세요."
    )
print(f"  OK: 설정된 카메라 모두 감지됨 ({len(CAM_SERIALS)}개)")
print("-" * 60)

# (B) 기존 파이프라인 정리
if "_rs_pipelines" in globals():
    for _p in _rs_pipelines.values():
        try:
            _p.stop()
        except Exception:
            pass
    print("[cleanup] 기존 RS 파이프라인 정리 완료")

gc.collect()
time.sleep(1.0)

# (C) RealSense 파이프라인 시작 (순차 + retry)
_rs_pipelines: dict[str, rs.pipeline] = {}

for cam_name, serial in CAM_SERIALS.items():
    for attempt in range(1, 4):
        try:
            pipe = rs.pipeline()
            cfg = rs.config()
            cfg.enable_device(serial)
            cfg.enable_stream(rs.stream.color, IMG_WIDTH, IMG_HEIGHT, rs.format.bgr8, 30)
            pipe.start(cfg)
            # warm-up
            for _ in range(10):
                pipe.wait_for_frames(timeout_ms=5000)
            _rs_pipelines[cam_name] = pipe
            print(f"[camera] {cam_name} ({serial}) started (attempt {attempt}): {IMG_WIDTH}x{IMG_HEIGHT}@30fps")
            break
        except Exception as _e:
            try:
                pipe.stop()
            except Exception:
                pass
            if attempt == 3:
                for _pp in _rs_pipelines.values():
                    try:
                        _pp.stop()
                    except Exception:
                        pass
                raise RuntimeError(f"[camera] '{cam_name}' ({serial}) 시작 실패: {_e}") from _e
            print(f"  [RETRY {attempt}/3] {cam_name}: {_e}")
            time.sleep(2.0)
    time.sleep(0.5)  # USB settle

# (D) 로봇 연결 (state 읽기용)
robot_obs = rby.create_robot(ROBOT_IP, "a") if hasattr(rby, "create_robot") else rby.create_robot_a(ROBOT_IP)
robot_obs.connect()
assert robot_obs.is_connected(), f"Failed to connect robot at {ROBOT_IP}"
print(f"[robot] connected for observation: {ROBOT_IP}")


# ---------- observation helper ----------
def get_observation() -> dict:
    """Read cameras + robot state and return raw observation dict."""
    images = {}
    for cam_name, pipe in _rs_pipelines.items():
        frames = pipe.wait_for_frames(timeout_ms=5000)
        color_frame = frames.get_color_frame()
        if not color_frame:
            raise RuntimeError(f"[obs] no color frame from {cam_name}")
        bgr = np.asanyarray(color_frame.get_data())
        rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
        rgb_resized = resize_with_pad(rgb, RENDER_SIZE, RENDER_SIZE)
        images[cam_name] = rgb_resized

    q = np.asarray(robot_obs.get_state().position, dtype=np.float64)
    right_arm = q[RIGHT_SLICE]
    left_arm = q[LEFT_SLICE]

    if USE_REMOTE_GRIPPER and "gripper_obj" in globals() and gripper_obj is not None:
        try:
            _gnorm = gripper_obj.get_normalized_target()
            right_grip = float(_gnorm[0])
            left_grip = float(_gnorm[1])
        except Exception:
            right_grip = 1.0
            left_grip = 1.0
    else:
        right_grip = 1.0
        left_grip = 1.0

    state16 = np.concatenate([
        right_arm,
        left_arm,
        np.array([right_grip]),
        np.array([left_grip]),
    ]).astype(np.float32)

    return {
        "head_image": images["head"],
        "right_wrist_image": images["right_wrist"],
        "state": state16,
    }


# (E) 첫 관측 테스트 + 시각화
obs = get_observation()
print(f"[obs] head_image: {obs['head_image'].shape}, right_wrist_image: {obs['right_wrist_image'].shape}")
print(f"[obs] state: {obs['state'].shape}")
print(f"[obs] state (rad): {np.round(obs['state'], 4)}")

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
for ax, (title, img) in zip(axes, [("head", obs["head_image"]), ("right_wrist", obs["right_wrist_image"])]):
    ax.imshow(img)
    ax.set_title(title, fontsize=11)
    ax.axis("off")
fig.suptitle("Initial Observation (LAP)", fontsize=13)
plt.tight_layout()
plt.show()

ENV_SETUP_DONE = True
print("[setup] 완료: 카메라 + 로봇 state 관측 확인")


## Step 4 — Dry-run Inference (로봇 명령 없음)

LAP inference를 1회 수행하고 출력만 확인합니다. 로봇에는 아무 명령도 보내지 않습니다.

**Joint 모드 출력값 해석:**
- `parse_lap_action()` 반환값은 **absolute joint target (rad)** — shape `(action_horizon, 8)`
- 현재 state와 가깝게 나오면 정상 (delta가 작을수록 현재 위치 근처)
- Index 0-6: right arm joints, Index 7: right gripper (normalized 0-1)

**EEF 모드 출력값 해석:**
- `parse_lap_action()` 반환값은 **absolute EEF** — shape `(action_horizon, 7)`
- Index 0-2: `[dx, dy, dz]` (m), Index 3-5: `[droll, dpitch, dyaw]` (rad), Index 6: gripper (0-1)
- 모델 출력 absolute EEF target 포즈 표시

In [ ]:
# ---------- Step 4: Dry-run inference ----------
if not globals().get("ENV_SETUP_DONE", False):
    raise RuntimeError("먼저 Step 3 셀을 실행하세요.")

obs_dry = get_observation()
lap_input = build_lap_input(
    state16=obs_dry["state"],
    head_image_hwc=obs_dry["head_image"],
    right_wrist_image_hwc=obs_dry["right_wrist_image"],
    prompt=PROMPT,
)

result_dry = policy.infer(lap_input)
actions_dry = parse_lap_action(result_dry)

current_state = obs_dry["state"]
print(f"[dry-run] IS_EEF_MODE={IS_EEF_MODE}")
print(f"[dry-run] action chunk shape: {actions_dry.shape}")

if IS_EEF_MODE:
    # --- EEF 모드 dry-run ---
    current_eef = compute_eef_from_joints(current_state[:7], float(current_state[14]))
    print(f"[dry-run] current EEF (FK): pos={np.round(current_eef[:3], 4)} euler={np.round(current_eef[3:6], 4)} grip={current_eef[6]:.3f}")
    print()
    print(f"[dry-run] model EEF target[0] (abs): pos={np.round(actions_dry[0, :3], 5)} euler={np.round(actions_dry[0, 3:6], 5)} grip={actions_dry[0, 6]:.4f}")
    print(f"[dry-run] model EEF target[-1] (abs): pos={np.round(actions_dry[-1, :3], 5)} euler={np.round(actions_dry[-1, 3:6], 5)} grip={actions_dry[-1, 6]:.4f}")
    print()

    # Model outputs absolute EEF targets (not deltas)
    target_eef_first = actions_dry[0].copy()
    target_eef_last = actions_dry[-1].copy()
    print(f"[dry-run] target EEF[0] : pos={np.round(target_eef_first[:3], 4)} euler={np.round(target_eef_first[3:6], 4)}")
    print(f"[dry-run] target EEF[-1]: pos={np.round(target_eef_last[:3], 4)} euler={np.round(target_eef_last[3:6], 4)}")
    print()

    # absolute target norms (sanity check — should be ~0.3-0.7 m)
    print("[dry-run] absolute EEF targets per step:")
    for t in range(actions_dry.shape[0]):
        pos_norm = float(np.linalg.norm(actions_dry[t, :3]))
        rot_norm = float(np.linalg.norm(actions_dry[t, 3:6]))
        if t == 0 or t == actions_dry.shape[0] - 1 or (t + 1) % 4 == 0:
            print(f"  step {t:2d}: abs_pos={pos_norm:.5f} m | abs_euler={rot_norm:.5f} rad | gripper={actions_dry[t, 6]:.3f}")
else:
    # --- Joint 모드 dry-run (기존 로직) ---
    print(f"[dry-run] current state (rad): {np.round(current_state, 4)}")
    print()
    print(f"[dry-run] absolute right_arm[0] : {np.round(actions_dry[0, :7], 4)}")
    print(f"[dry-run] absolute right_arm[-1]: {np.round(actions_dry[-1, :7], 4)}")
    print(f"[dry-run] gripper[0]={actions_dry[0, 7]:.4f}, gripper[-1]={actions_dry[-1, 7]:.4f}")
    print()

    current_right_arm = current_state[:7]
    delta_arm = actions_dry[:, :7] - current_right_arm[None, :]
    print("[dry-run] implied delta from current right_arm state:")
    for t in range(actions_dry.shape[0]):
        d_norm = float(np.linalg.norm(delta_arm[t]))
        if t == 0 or t == actions_dry.shape[0] - 1 or (t + 1) % 4 == 0:
            print(f"  step {t:2d}: delta_norm={d_norm:.4f} rad | gripper={actions_dry[t, 7]:.3f}")

---
## [선택] 시뮬레이터 초기 자세 이동

`SIM_IP`에서 실행 중인 시뮬레이터 로봇을 실로봇과 동일한 초기 자세로 이동시킵니다.
실로봇 inference 전에 시뮬레이터로 먼저 동작을 검증할 때 사용합니다.

> `TEST_MODE=False`이면 자동으로 건너뜁니다.

In [ ]:
# ---------- 시뮬레이터 initial pose 이동 ----------
if not globals().get("TEST_MODE", False):
    print("[sim-init] TEST_MODE=False -> skip")
else:
    SIM_ADDR = SIM_IP
    SIM_INIT_DT = 0.05
    SIM_INIT_HOLD_TIME = 0.5

    TORSO_INIT_DEG_SIM = np.array([0.0, 20.0, -40.0, 35.0, 0.0, 0.0], dtype=np.float64)
    INIT_RIGHT_DEG_SIM = np.array([-24.0, -60.0, 10.0, -120.0, -60.0, 85.0, 0.0], dtype=np.float64)
    INIT_LEFT_DEG_SIM  = np.array([-24.0,  60.0, -10.0, -120.0,  60.0, 85.0, 0.0], dtype=np.float64)
    MID1_RIGHT_DEG_SIM = np.array([70.0, -30.0, 0.0, -100.0, 0.0, -70.0, 0.0], dtype=np.float64)
    MID1_LEFT_DEG_SIM  = np.array([70.0,  30.0, 0.0, -100.0, 0.0, -70.0, 0.0], dtype=np.float64)
    MID2_RIGHT_DEG_SIM = np.array([0.0,  -15.0, 0.0, -100.0, 0.0, -70.0, 0.0], dtype=np.float64)
    MID2_LEFT_DEG_SIM  = np.array([0.0,   15.0, 0.0, -100.0, 0.0, -70.0, 0.0], dtype=np.float64)

    TORSO_INIT_SIM = np.deg2rad(TORSO_INIT_DEG_SIM)
    INIT_RIGHT_SIM = np.deg2rad(INIT_RIGHT_DEG_SIM)
    INIT_LEFT_SIM  = np.deg2rad(INIT_LEFT_DEG_SIM)
    MID1_RIGHT_SIM = np.deg2rad(MID1_RIGHT_DEG_SIM)
    MID1_LEFT_SIM  = np.deg2rad(MID1_LEFT_DEG_SIM)
    MID2_RIGHT_SIM = np.deg2rad(MID2_RIGHT_DEG_SIM)
    MID2_LEFT_SIM  = np.deg2rad(MID2_LEFT_DEG_SIM)

    TORSO_SLICE_SIM = slice(2, 8)
    RIGHT_SLICE_SIM = slice(8, 15)
    LEFT_SLICE_SIM  = slice(15, 22)

    def _build_body_cmd_sim(torso_q, right_q, left_q, dt):
        body_builder = rby.BodyComponentBasedCommandBuilder()
        if torso_q is not None:
            body_builder = body_builder.set_torso_command(
                rby.JointPositionCommandBuilder()
                .set_command_header(rby.CommandHeaderBuilder().set_control_hold_time(SIM_INIT_HOLD_TIME))
                .set_position(np.asarray(torso_q, dtype=np.float64))
                .set_minimum_time(float(dt))
            )
        body_builder = (
            body_builder
            .set_right_arm_command(
                rby.JointPositionCommandBuilder()
                .set_command_header(rby.CommandHeaderBuilder().set_control_hold_time(SIM_INIT_HOLD_TIME))
                .set_position(np.asarray(right_q, dtype=np.float64))
                .set_minimum_time(float(dt))
            )
            .set_left_arm_command(
                rby.JointPositionCommandBuilder()
                .set_command_header(rby.CommandHeaderBuilder().set_control_hold_time(SIM_INIT_HOLD_TIME))
                .set_position(np.asarray(left_q, dtype=np.float64))
                .set_minimum_time(float(dt))
            )
        )
        return rby.RobotCommandBuilder().set_command(
            rby.ComponentBasedCommandBuilder().set_body_command(body_builder)
        )

    def _move_waypoint_sim(robot, stream, torso_target, right_target, left_target, duration):
        q0 = np.asarray(robot.get_state().position, dtype=np.float64)
        start_torso = q0[TORSO_SLICE_SIM].copy()
        start_right = q0[RIGHT_SLICE_SIM].copy()
        start_left  = q0[LEFT_SLICE_SIM].copy()
        goal_torso = torso_target if torso_target is not None else start_torso
        steps = max(2, int(np.ceil(duration / SIM_INIT_DT)))
        torso_path = np.linspace(start_torso, goal_torso, num=steps)
        right_path = np.linspace(start_right, right_target, num=steps)
        left_path  = np.linspace(start_left,  left_target,  num=steps)
        for i in range(steps):
            cmd_wp = _build_body_cmd_sim(torso_path[i], right_path[i], left_path[i], dt=SIM_INIT_DT)
            try:
                stream.send_command(cmd_wp)
            except RuntimeError as exc:
                if "expired" in str(exc).lower():
                    robot.wait_for_control_ready(1000)
                    stream = robot.create_command_stream(priority=10)
                    stream.send_command(cmd_wp)
                else:
                    raise
            time.sleep(SIM_INIT_DT)
        return stream

    robot_sim = rby.create_robot_a(SIM_ADDR) if hasattr(rby, "create_robot_a") else rby.create_robot(SIM_ADDR, "a")
    robot_sim.connect()
    assert robot_sim.is_connected(), f"Failed to connect simulator at {SIM_ADDR}"

    robot_sim.power_on(".*")
    robot_sim.servo_on(".*")
    robot_sim.reset_fault_control_manager()
    if not robot_sim.enable_control_manager():
        raise RuntimeError("[sim-init] Failed to enable control manager")

    stream_sim = robot_sim.create_command_stream(priority=10)
    sim_waypoints = [
        (None,           MID1_RIGHT_SIM, MID1_LEFT_SIM, 5.0),
        (None,           MID2_RIGHT_SIM, MID2_LEFT_SIM, 5.0),
        (TORSO_INIT_SIM, INIT_RIGHT_SIM, INIT_LEFT_SIM, 3.0),
    ]
    for torso_t, right_t, left_t, dur_t in sim_waypoints:
        stream_sim = _move_waypoint_sim(robot_sim, stream_sim, torso_t, right_t, left_t, dur_t)

    time.sleep(0.2)
    q_init = np.asarray(robot_sim.get_state().position, dtype=np.float64)
    print("[sim-init] q_init right:", np.round(q_init[8:15], 4))
    print("[sim-init] q_init left :", np.round(q_init[15:22], 4))

    try:
        robot_sim.cancel_control()
    except Exception:
        pass
    if hasattr(robot_sim, "disconnect"):
        robot_sim.disconnect()

    SIM_INITIAL_POSE_DONE = True
    print("[sim-init] done")

## [선택] 시뮬레이터 단일 Action Chunk 전송 테스트

LAP inference를 1회 수행하고, action chunk를 시뮬레이터로 전송합니다.
실로봇에 전송하기 전 policy 출력이 안전한지 확인하는 용도입니다.

- **Joint 모드**: right arm(7D) absolute targets → `JointPositionCommandBuilder`
- **EEF 모드**: right arm absolute EEF(7D) → `CartesianCommandBuilder`
- Left arm은 현재 위치 유지
- 시뮬레이터에는 그리퍼 명령을 전송하지 않습니다 (팔 궤적 확인용)

> `TEST_MODE=False`이면 자동으로 건너뜁니다.

In [ ]:
# ---------- LAP action chunk -> 시뮬레이터 전송 ----------
if not globals().get("TEST_MODE", False):
    print("[sim-chunk] TEST_MODE=False -> skip")
else:
    if not globals().get("ENV_SETUP_DONE", False):
        raise RuntimeError("먼저 Step 3 셀을 실행하세요.")
    if not globals().get("SIM_INITIAL_POSE_DONE", False):
        raise RuntimeError("먼저 시뮬레이터 initial pose 셀을 실행하세요.")

    SIM_ADDR            = SIM_IP
    SIM_CHUNK_DT        = 0.1
    SIM_CHUNK_HOLD_TIME = 0.2
    SIM_CHUNK_PRIORITY  = 10

    # 1) inference (실로봇 카메라 관측 사용)
    obs_sim = get_observation()
    lap_input_sim = build_lap_input(
        state16=obs_sim["state"],
        head_image_hwc=obs_sim["head_image"],
        right_wrist_image_hwc=obs_sim["right_wrist_image"],
        prompt=PROMPT,
    )
    result_sim = policy.infer(lap_input_sim)
    actions_sim = parse_lap_action(result_sim)

    print(f"[sim-chunk] IS_EEF_MODE={IS_EEF_MODE}")
    print(f"[sim-chunk] action chunk shape: {actions_sim.shape}")

    if IS_EEF_MODE:
        print(f"[sim-chunk] model EEF target[0] (abs): {np.round(actions_sim[0], 5)}")
        print(f"[sim-chunk] model EEF target[-1] (abs): {np.round(actions_sim[-1], 5)}")
        pos_range = float(np.linalg.norm(actions_sim[-1, :3] - actions_sim[0, :3]))
        print(f"[sim-chunk] EEF trajectory range (last-first): {pos_range:.5f} m")
    else:
        right_targets_preview = actions_sim[:, :7]
        right_gripper_preview = actions_sim[:, 7]
        print(f"[sim-chunk] right target[0] : {np.round(right_targets_preview[0], 4)}")
        print(f"[sim-chunk] right target[-1]: {np.round(right_targets_preview[-1], 4)}")
        print(f"[sim-chunk] gripper[0]={right_gripper_preview[0]:.4f}, gripper[-1]={right_gripper_preview[-1]:.4f}")
        total_cmd_norm_r = float(np.linalg.norm(right_targets_preview[-1] - right_targets_preview[0]))
        print(f"[sim-chunk] cmd movement norm | right={total_cmd_norm_r:.4f}")
        if total_cmd_norm_r < 1e-3:
            print("[sim-chunk][WARN] 모델 output 변화가 매우 작습니다.")

    # 2) 시뮬레이터 연결
    robot_sim = rby.create_robot_a(SIM_ADDR) if hasattr(rby, "create_robot_a") else rby.create_robot(SIM_ADDR, "a")
    robot_sim.connect()
    assert robot_sim.is_connected()

    robot_sim.power_on(".*")
    robot_sim.servo_on(".*")
    robot_sim.reset_fault_control_manager()
    if not robot_sim.enable_control_manager():
        raise RuntimeError("[sim-chunk] Failed to enable control manager")

    q0_sim      = np.asarray(robot_sim.get_state().position, dtype=np.float64)
    torso_hold  = q0_sim[2:8].copy()
    left_hold   = q0_sim[15:22].copy()
    start_r_sim = q0_sim[8:15].copy()

    # 3) stream 전송
    stream_sim  = robot_sim.create_command_stream(priority=SIM_CHUNK_PRIORITY)
    send_errors = 0

    for i in range(actions_sim.shape[0]):
        if IS_EEF_MODE:
            # EEF 모드: model output absolute target → SE3 → CartesianCommand
            target_eef = actions_sim[i].copy()
            T_target = eef_7d_to_se3(target_eef)
            cmd = build_cartesian_right_arm_cmd(
                torso_hold=torso_hold, T_target=T_target, left_hold=left_hold,
                hold_time=SIM_CHUNK_HOLD_TIME,
            )
        else:
            # Joint 모드
            cmd = build_joint_right_arm_cmd(
                torso_hold=torso_hold, right_target=actions_sim[i, :7],
                left_hold=left_hold, dt=SIM_CHUNK_DT, hold_time=SIM_CHUNK_HOLD_TIME,
            )

        try:
            stream_sim.send_command(cmd)
        except RuntimeError as exc:
            send_errors += 1
            if "expired" in str(exc).lower():
                robot_sim.wait_for_control_ready(1000)
                stream_sim = robot_sim.create_command_stream(priority=SIM_CHUNK_PRIORITY)
                stream_sim.send_command(cmd)
            else:
                raise

        if i == 0 or (i + 1) % 5 == 0 or (i + 1) == actions_sim.shape[0]:
            q_now = np.asarray(robot_sim.get_state().position, dtype=np.float64)
            moved_r = float(np.linalg.norm(q_now[8:15] - start_r_sim))
            if IS_EEF_MODE:
                now_eef = compute_eef_from_joints(q_now[8:15], 1.0)
                pos_err = float(np.linalg.norm(target_eef[:3] - now_eef[:3]))
                print(
                    f"[sim-chunk] step {i+1:3d}/{actions_sim.shape[0]} "
                    f"| eef_pos_err={pos_err:.5f} m "
                    f"| joint_moved={moved_r:.4f}"
                )
            else:
                err_r = float(np.linalg.norm(actions_sim[i, :7] - q_now[8:15]))
                print(
                    f"[sim-chunk] step {i+1:3d}/{actions_sim.shape[0]} "
                    f"| tracking_err(r={err_r:.4f}) "
                    f"| moved(r={moved_r:.4f})"
                )

        time.sleep(SIM_CHUNK_DT)

    time.sleep(0.5)
    q_after = np.asarray(robot_sim.get_state().position, dtype=np.float64)
    obs_norm_r = float(np.linalg.norm(q_after[8:15] - start_r_sim))
    print(f"\n[sim-chunk] === 결과 ===")
    print(f"[sim-chunk] observed movement norm | right={obs_norm_r:.4f}")
    print(f"[sim-chunk] send errors: {send_errors}")
    if obs_norm_r < 1e-3:
        print("[sim-chunk][WARN] 실제 이동량이 거의 없음. 관측 형식을 확인하세요.")

    if IS_EEF_MODE:
        final_eef = compute_eef_from_joints(q_after[8:15], 1.0)
        init_eef = compute_eef_from_joints(start_r_sim, 1.0)
        print(f"[sim-chunk] EEF pos moved: {float(np.linalg.norm(final_eef[:3] - init_eef[:3])):.4f} m")

    try:
        robot_sim.cancel_control()
    except Exception:
        pass
    if hasattr(robot_sim, "disconnect"):
        robot_sim.disconnect()

---
## [선택] 시뮬레이터 Episode 루프

실로봇 카메라 관측으로 LAP inference를 반복 수행하고, action chunk를 시뮬레이터에 전송합니다.
실로봇 episode loop와 동일한 구조이지만 명령 대상이 시뮬레이터입니다.

- **관측(observation)**: 실로봇 카메라 + 실로봇 state (Step 3에서 구성)
- **명령(command)**: 시뮬레이터 (`SIM_IP`)
- **Joint 모드**: Right arm absolute target → `JointPositionCommandBuilder`
- **EEF 모드**: FK + delta → target SE3 → `CartesianCommandBuilder`
- Left arm: 시뮬레이터 현재 위치 유지 (hold)
- 그리퍼: 시뮬레이터에는 전송하지 않음

> `TEST_MODE=False`이면 자동으로 건너뜁니다.

In [ ]:
# ---------- [선택] 시뮬레이터 Episode 루프 — LAP (Right Arm Only) ----------
if not globals().get("TEST_MODE", False):
    print("[sim-replay] TEST_MODE=False -> skip")
else:
    if not globals().get("ENV_SETUP_DONE", False):
        raise RuntimeError("먼저 Step 3 셀을 실행하세요.")
    if not globals().get("SIM_INITIAL_POSE_DONE", False):
        raise RuntimeError("먼저 시뮬레이터 initial pose 셀을 실행하세요.")

    # --------------------------------------------------
    # 시뮬레이터 재생 파라미터
    # --------------------------------------------------
    SIM_REPLAY_DT        = 0.1
    SIM_REPLAY_PRIORITY  = 10
    SIM_REPLAY_HOLD_TIME = 0.2
    SIM_EPISODE_LENGTH   = EPISODE_LENGTH

    # --------------------------------------------------
    # 시뮬레이터 연결
    # --------------------------------------------------
    robot_sim = rby.create_robot_a(SIM_IP) if hasattr(rby, "create_robot_a") else rby.create_robot(SIM_IP, "a")
    robot_sim.connect()
    assert robot_sim.is_connected(), f"Failed to connect simulator at {SIM_IP}"

    robot_sim.power_on(".*")
    robot_sim.servo_on(".*")
    robot_sim.reset_fault_control_manager()
    if not robot_sim.enable_control_manager():
        robot_sim.disconnect()
        raise RuntimeError("[sim-replay] Failed to enable control manager")

    q_init_sim   = np.asarray(robot_sim.get_state().position, dtype=np.float64)
    ep_start_r   = q_init_sim[8:15].copy()
    torso_hold   = q_init_sim[2:8].copy()
    left_hold    = q_init_sim[15:22].copy()
    print(f"[sim-replay] IS_EEF_MODE={IS_EEF_MODE}")
    print(f"[sim-replay] SIM_EPISODE_LENGTH={SIM_EPISODE_LENGTH}, EXECUTE_CHUNK_SIZE={EXECUTE_CHUNK_SIZE}")
    print(f"[sim-replay] episode start | right={np.round(ep_start_r, 4)}")

    stream_sim   = robot_sim.create_command_stream(priority=SIM_REPLAY_PRIORITY)
    total_steps  = 0
    chunk_idx    = 0
    send_errors  = 0

    # 로깅 버퍼
    _sim_log_cmd_right    = []
    _sim_log_cmd_grip     = []
    _sim_log_actual_right = []
    _sim_log_actual_step  = []
    _sim_log_chunk_start  = []

    # --------------------------------------------------
    # Episode loop: observe (실로봇) -> infer -> send to sim
    # --------------------------------------------------
    while total_steps < SIM_EPISODE_LENGTH:
        remaining = SIM_EPISODE_LENGTH - total_steps

        # 1) 실로봇 관측 + inference
        obs_ep = get_observation()
        lap_input_ep = build_lap_input(
            state16=obs_ep["state"],
            head_image_hwc=obs_ep["head_image"],
            right_wrist_image_hwc=obs_ep["right_wrist_image"],
            prompt=PROMPT,
        )
        result_ep = policy.infer(lap_input_ep)
        actions_ep = parse_lap_action(result_ep)

        # left arm hold (시뮬레이터 현재 위치)
        q_sim_now  = np.asarray(robot_sim.get_state().position, dtype=np.float64)
        left_hold  = q_sim_now[15:22].copy()
        torso_hold = q_sim_now[2:8].copy()

        chunk_size    = actions_ep.shape[0]
        execute_size  = EXECUTE_CHUNK_SIZE if EXECUTE_CHUNK_SIZE is not None else chunk_size
        steps_to_send = min(execute_size, remaining)

        _sim_log_chunk_start.append(total_steps)

        if IS_EEF_MODE:
            pos_norms = np.linalg.norm(actions_ep[:steps_to_send, :3], axis=1)
            print(
                f"[sim-replay] chunk {chunk_idx+1} | "
                f"steps {total_steps+1}~{total_steps+steps_to_send}/{SIM_EPISODE_LENGTH} "
                f"(execute {steps_to_send}/{chunk_size}) | "
                f"pos_abs[0]={pos_norms[0]:.5f} m"
            )
        else:
            right_targets = actions_ep[:, :7]
            right_gripper_out = actions_ep[:, 7]
            total_cmd_norm = float(np.linalg.norm(
                right_targets[min(steps_to_send-1, chunk_size-1)] - right_targets[0]
            ))
            print(
                f"[sim-replay] chunk {chunk_idx+1} | "
                f"steps {total_steps+1}~{total_steps+steps_to_send}/{SIM_EPISODE_LENGTH} "
                f"(execute {steps_to_send}/{chunk_size}) | cmd_norm={total_cmd_norm:.4f}"
            )

        # 2) chunk 전송
        for i in range(steps_to_send):
            idx = min(i, chunk_size - 1)

            if IS_EEF_MODE:
                q_step_sim = np.asarray(robot_sim.get_state().position, dtype=np.float64)
                target_eef = actions_ep[idx].copy()
                T_target = eef_7d_to_se3(target_eef)
                cmd = build_cartesian_right_arm_cmd(
                    torso_hold=torso_hold, T_target=T_target, left_hold=left_hold,
                    hold_time=SIM_REPLAY_HOLD_TIME,
                )
                gripper_val = float(actions_ep[idx, 6])
            else:
                cmd = build_joint_right_arm_cmd(
                    torso_hold=torso_hold, right_target=right_targets[idx],
                    left_hold=left_hold, dt=SIM_REPLAY_DT, hold_time=SIM_REPLAY_HOLD_TIME,
                )
                gripper_val = float(right_gripper_out[idx])

            try:
                stream_sim.send_command(cmd)
            except RuntimeError as exc:
                send_errors += 1
                if "expired" in str(exc).lower():
                    print(f"[sim-replay][WARN] stream expired at step {total_steps}")
                    robot_sim.wait_for_control_ready(1000)
                    stream_sim = robot_sim.create_command_stream(priority=SIM_REPLAY_PRIORITY)
                    stream_sim.send_command(cmd)
                else:
                    raise

            _sim_log_cmd_right.append(
                actions_ep[idx].copy() if IS_EEF_MODE else right_targets[idx].copy()
            )
            _sim_log_cmd_grip.append(gripper_val)

            if i == 0 or (i + 1) % 10 == 0 or (i + 1) == steps_to_send:
                q_diag  = np.asarray(robot_sim.get_state().position, dtype=np.float64)
                moved_r = float(np.linalg.norm(q_diag[8:15] - ep_start_r))
                _sim_log_actual_right.append(q_diag[8:15].copy())
                _sim_log_actual_step.append(total_steps + i)

                if IS_EEF_MODE:
                    now_eef = compute_eef_from_joints(q_diag[8:15], 1.0)
                    pos_err = float(np.linalg.norm(target_eef[:3] - now_eef[:3]))
                    print(
                        f"  step {total_steps+i+1:4d}/{SIM_EPISODE_LENGTH} "
                        f"| eef_pos_err={pos_err:.5f} m "
                        f"| joint_moved={moved_r:.4f} "
                        f"| gripper={gripper_val:+.3f}"
                    )
                else:
                    err_r = float(np.linalg.norm(right_targets[idx] - q_diag[8:15]))
                    print(
                        f"  step {total_steps+i+1:4d}/{SIM_EPISODE_LENGTH} "
                        f"| tracking_err(r={err_r:.4f}) "
                        f"| moved(r={moved_r:.4f}) "
                        f"| gripper={gripper_val:+.3f}"
                    )

            time.sleep(SIM_REPLAY_DT)

        total_steps += steps_to_send
        chunk_idx   += 1

    # --------------------------------------------------
    # 최종 결과
    # --------------------------------------------------
    time.sleep(0.5)
    q_after    = np.asarray(robot_sim.get_state().position, dtype=np.float64)
    obs_norm_r = float(np.linalg.norm(q_after[8:15] - ep_start_r))

    print(f"\n[sim-replay] === episode done ===")
    print(f"[sim-replay] IS_EEF_MODE={IS_EEF_MODE}")
    print(f"[sim-replay] total_steps={total_steps}, chunks={chunk_idx}, send_errors={send_errors}")
    print(f"[sim-replay] ep_start right : {np.round(ep_start_r, 4)}")
    print(f"[sim-replay] q_after  right : {np.round(q_after[8:15], 4)}")
    print(f"[sim-replay] total movement norm | right={obs_norm_r:.4f}")
    print(f"[sim-replay] log: {len(_sim_log_cmd_right)} steps, {len(_sim_log_chunk_start)} chunks")

    if IS_EEF_MODE:
        final_eef = compute_eef_from_joints(q_after[8:15], 1.0)
        init_eef = compute_eef_from_joints(ep_start_r, 1.0)
        print(f"[sim-replay] EEF pos moved: {float(np.linalg.norm(final_eef[:3] - init_eef[:3])):.4f} m")

    try:
        robot_sim.cancel_control()
    except Exception:
        pass
    if hasattr(robot_sim, "disconnect"):
        robot_sim.disconnect()

---
## Step 5 — 실로봇 Inference Episode 루프 실행

LAP inference → action chunk 실행을 반복하며 전체 에피소드를 수행합니다.

- **`EPISODE_LENGTH`**: 총 실행 스텝 수
- **`EXECUTE_CHUNK_SIZE`**: inference 1회당 실제 전송 스텝 수 (None = 전체 chunk)
- **`REPLAY_DT`**: 각 step 전송 간격(초)
- **`USE_RTC`**: RTC(Real-Time Chunking) 활성화 여부 — 서버도 `--use-rtc`로 시작해야 함
  - `RTC_EXECUTION_HORIZON`: 매 chunk 실제 실행 스텝 수 (= `EXECUTE_CHUNK_SIZE`)
  - `RTC_MAX_GUIDANCE_WEIGHT`: denoising 가이던스 최대 가중치
  - `RTC_SCHEDULE`: 가이던스 가중치 스케줄 (`linear` | `exp` | `ones` | `zeros`)

### Joint 모드 (`IS_EEF_MODE=False`)
- `parse_lap_action()` → shape `(action_horizon, 8)` — absolute joint target (rad)
- Index 0-6: right arm → `JointPositionCommandBuilder`
- Index 7: right gripper (normalized 0-1)
- Left arm: 현재 위치 유지

### EEF 모드 (`IS_EEF_MODE=True`)
- `parse_lap_action()` → shape `(action_horizon, 7)` — absolute EEF
- Index 0-5: `[x,y,z,r,p,y]` → absolute EEF target → SE3
- Index 6: gripper (normalized 0-1)
- Right arm: `CartesianCommandBuilder` (SDK 내부 QP IK)
- Left arm: 현재 위치 유지

In [ ]:
# ---------- Step 5: LAP policy action replay — episode loop ----------
if not globals().get("ENV_SETUP_DONE", False):
    raise RuntimeError("먼저 Step 3 셀을 실행하세요.")
if not globals().get("INITIAL_POSE_DONE", False):
    raise RuntimeError("먼저 Step 2 셀(초기 자세 이동)을 실행하세요.")

# --------------------------------------------------
# 재생 파라미터
# --------------------------------------------------
REPLAY_DT        = 0.05
REPLAY_PRIORITY  = ARM_COMMAND_PRIORITY
REPLAY_HOLD_TIME = 0.2

# EEF 모드 CartesianCommand 파라미터
EEF_TRANSLATION_WEIGHT = 1.0
EEF_ROTATION_WEIGHT    = 1.0
EEF_VELOCITY_SCALING   = 0.8

# --------------------------------------------------
# 로봇 연결
# --------------------------------------------------
robot = rby.create_robot(ROBOT_IP, "a") if hasattr(rby, "create_robot") else rby.create_robot_a(ROBOT_IP)
robot.connect()
assert robot.is_connected(), f"Failed to connect robot at {ROBOT_IP}"

robot.power_on(".*")
robot.servo_on(".*")
robot.reset_fault_control_manager()
if not robot.enable_control_manager():
    robot.disconnect()
    raise RuntimeError("[real-replay] Failed to enable control manager")

q_init     = np.asarray(robot.get_state().position, dtype=np.float64)
ep_start_r = q_init[RIGHT_SLICE].copy()
ep_start_l = q_init[LEFT_SLICE].copy()
torso_hold = q_init[TORSO_SLICE].copy()
print(f"[real-replay] IS_EEF_MODE={IS_EEF_MODE}")
print(f"[real-replay] EPISODE_LENGTH={EPISODE_LENGTH}, EXECUTE_CHUNK_SIZE={EXECUTE_CHUNK_SIZE}")
print(f"[real-replay] episode start | right={np.round(ep_start_r, 4)}")

if IS_EEF_MODE:
    # EEF 모드: 초기 FK 포즈 확인
    _init_eef = compute_eef_from_joints(ep_start_r, 1.0)
    print(f"[real-replay] initial EEF (FK): pos={np.round(_init_eef[:3], 4)} euler={np.round(_init_eef[3:6], 4)}")

stream      = robot.create_command_stream(priority=REPLAY_PRIORITY)
total_steps = 0
chunk_idx   = 0
send_errors = 0

# --------------------------------------------------
# 로깅 버퍼
# --------------------------------------------------
_log_cmd_right    = []
_log_cmd_grip     = []
_log_actual_right = []
_log_actual_step  = []
_log_chunk_start  = []
_IMG_KEYS = ["head_image", "right_wrist_image"]
_log_obs_images   = []
# EEF 모드 추가 로깅
_log_eef_target   = []  # target EEF 7D per step
_log_eef_current  = []  # current EEF 7D per observation

# --------------------------------------------------
# RTC: 에피소드 시작 전 서버 측 history 초기화
if USE_RTC:
    policy.infer({"__rtc_reset__": True})
    print("[real-replay] RTC enabled — server policy state reset")

# Episode loop: observe -> infer -> send right arm (left arm hold)
# --------------------------------------------------
while total_steps < EPISODE_LENGTH:
    remaining = EPISODE_LENGTH - total_steps

    # 1) observation + inference
    obs_ep = get_observation()

    _img_snap = {"step": total_steps}
    for _k in _IMG_KEYS:
        if _k in obs_ep:
            _img_snap[_k] = obs_ep[_k].copy()
    _log_obs_images.append(_img_snap)

    lap_input_ep = build_lap_input(
        state16=obs_ep["state"],
        head_image_hwc=obs_ep["head_image"],
        right_wrist_image_hwc=obs_ep["right_wrist_image"],
        prompt=PROMPT,
    )
    result_ep = policy.infer(lap_input_ep)
    actions_ep = parse_lap_action(result_ep)

    # 2) hold positions
    q_now = np.asarray(robot.get_state().position, dtype=np.float64)
    left_hold = q_now[LEFT_SLICE].copy()
    torso_hold = q_now[TORSO_SLICE].copy()

    chunk_size    = actions_ep.shape[0]
    execute_size  = EXECUTE_CHUNK_SIZE if EXECUTE_CHUNK_SIZE is not None else chunk_size
    steps_to_send = min(execute_size, remaining)

    _log_chunk_start.append(total_steps)

    if IS_EEF_MODE:
        # === EEF 모드 ===
        # EEF delta norms for logging
        pos_norms = np.linalg.norm(actions_ep[:steps_to_send, :3], axis=1)
        print(
            f"[real-replay] chunk {chunk_idx+1} | "
            f"steps {total_steps+1}~{total_steps+steps_to_send}/{EPISODE_LENGTH} "
            f"(execute {steps_to_send}/{chunk_size}) | "
            f"pos_abs[0]={pos_norms[0]:.5f} m"
        )

        # 현재 EEF 포즈 (매 chunk 시작 시 fresh FK)
        current_right_arm = q_now[RIGHT_SLICE].copy()
        if USE_REMOTE_GRIPPER and globals().get("gripper_obj") is not None:
            try:
                _gnorm = gripper_obj.get_normalized_target()
                current_grip = float(_gnorm[0])
            except Exception:
                current_grip = 1.0
        else:
            current_grip = 1.0
        current_eef = compute_eef_from_joints(current_right_arm, current_grip)
        _log_eef_current.append(current_eef.copy())

        for i in range(steps_to_send):
            idx = min(i, chunk_size - 1)

            # 매 step: model output absolute EEF target → SE3
            q_step = np.asarray(robot.get_state().position, dtype=np.float64)
            step_right_arm = q_step[RIGHT_SLICE].copy()
            target_eef = actions_ep[idx].copy()
            T_target = eef_7d_to_se3(target_eef)

            cmd = build_cartesian_right_arm_cmd(
                torso_hold=torso_hold,
                T_target=T_target,
                left_hold=left_hold,
                hold_time=REPLAY_HOLD_TIME,
                translation_weight=EEF_TRANSLATION_WEIGHT,
                rotation_weight=EEF_ROTATION_WEIGHT,
                velocity_scaling=EEF_VELOCITY_SCALING,
            )

            try:
                stream.send_command(cmd)
            except RuntimeError as exc:
                send_errors += 1
                if "expired" in str(exc).lower():
                    print(f"[real-replay][WARN] stream expired at step {total_steps}")
                    robot.wait_for_control_ready(1000)
                    stream = robot.create_command_stream(priority=REPLAY_PRIORITY)
                    stream.send_command(cmd)
                else:
                    raise

            # Gripper
            gripper_val = float(actions_ep[idx, 6])
            if USE_REMOTE_GRIPPER and globals().get("gripper_obj") is not None:
                try:
                    gripper_obj.set_normalized_target(
                        np.array([gripper_val, 1.0]), wait_for_reply=False,
                    )
                except Exception as _ge:
                    print(f"[real-replay][WARN] gripper send failed: {_ge}")

            _log_cmd_right.append(step_right_arm.copy())
            _log_cmd_grip.append(gripper_val)
            _log_eef_target.append(target_eef.copy())

            if i == 0 or (i + 1) % 10 == 0 or (i + 1) == steps_to_send:
                q_diag = np.asarray(robot.get_state().position, dtype=np.float64)
                diag_eef = compute_eef_from_joints(q_diag[RIGHT_SLICE], current_grip)
                pos_err = float(np.linalg.norm(target_eef[:3] - diag_eef[:3]))
                moved_r = float(np.linalg.norm(q_diag[RIGHT_SLICE] - ep_start_r))
                _log_actual_right.append(q_diag[RIGHT_SLICE].copy())
                _log_actual_step.append(total_steps + i)
                print(
                    f"  step {total_steps+i+1:4d}/{EPISODE_LENGTH} "
                    f"| eef_pos_err={pos_err:.5f} m "
                    f"| joint_moved={moved_r:.4f} "
                    f"| gripper={gripper_val:+.3f}"
                )

            time.sleep(REPLAY_DT)

    else:
        # === Joint 모드 (기존 로직) ===
        right_targets = actions_ep[:, :7]
        right_gripper_out = actions_ep[:, 7]

        total_cmd_norm = float(np.linalg.norm(
            right_targets[min(steps_to_send-1, chunk_size-1)] - right_targets[0]
        ))
        print(
            f"[real-replay] chunk {chunk_idx+1} | "
            f"steps {total_steps+1}~{total_steps+steps_to_send}/{EPISODE_LENGTH} "
            f"(execute {steps_to_send}/{chunk_size}) | cmd_norm={total_cmd_norm:.4f}"
        )

        for i in range(steps_to_send):
            idx = min(i, chunk_size - 1)

            cmd = build_joint_right_arm_cmd(
                torso_hold=torso_hold,
                right_target=right_targets[idx],
                left_hold=left_hold,
                dt=REPLAY_DT,
                hold_time=REPLAY_HOLD_TIME,
            )

            try:
                stream.send_command(cmd)
            except RuntimeError as exc:
                send_errors += 1
                if "expired" in str(exc).lower():
                    print(f"[real-replay][WARN] stream expired at step {total_steps}")
                    robot.wait_for_control_ready(1000)
                    stream = robot.create_command_stream(priority=REPLAY_PRIORITY)
                    stream.send_command(cmd)
                else:
                    raise

            if USE_REMOTE_GRIPPER and globals().get("gripper_obj") is not None:
                try:
                    gripper_obj.set_normalized_target(
                        np.array([float(right_gripper_out[idx]), 1.0]),
                        wait_for_reply=False,
                    )
                except Exception as _ge:
                    print(f"[real-replay][WARN] gripper send failed: {_ge}")

            _log_cmd_right.append(right_targets[idx].copy())
            _log_cmd_grip.append(float(right_gripper_out[idx]))

            if i == 0 or (i + 1) % 10 == 0 or (i + 1) == steps_to_send:
                q_diag  = np.asarray(robot.get_state().position, dtype=np.float64)
                err_r   = float(np.linalg.norm(right_targets[idx] - q_diag[RIGHT_SLICE]))
                moved_r = float(np.linalg.norm(q_diag[RIGHT_SLICE] - ep_start_r))
                _log_actual_right.append(q_diag[RIGHT_SLICE].copy())
                _log_actual_step.append(total_steps + i)
                print(
                    f"  step {total_steps+i+1:4d}/{EPISODE_LENGTH} "
                    f"| tracking_err(r={err_r:.4f}) "
                    f"| moved(r={moved_r:.4f}) "
                    f"| gripper={right_gripper_out[idx]:+.3f}"
                )

            time.sleep(REPLAY_DT)

    total_steps += steps_to_send
    chunk_idx   += 1

# --------------------------------------------------
# 최종 결과
# --------------------------------------------------
time.sleep(0.5)
q_after    = np.asarray(robot.get_state().position, dtype=np.float64)
obs_norm_r = float(np.linalg.norm(q_after[RIGHT_SLICE] - ep_start_r))

print(f"\n[real-replay] === episode done ===")
print(f"[real-replay] IS_EEF_MODE={IS_EEF_MODE}")
print(f"[real-replay] total_steps={total_steps}, chunks={chunk_idx}, send_errors={send_errors}")
print(f"[real-replay] ep_start right : {np.round(ep_start_r, 4)}")
print(f"[real-replay] q_after  right : {np.round(q_after[RIGHT_SLICE], 4)}")
print(f"[real-replay] total movement norm | right={obs_norm_r:.4f}")
print(f"[real-replay] log: {len(_log_cmd_right)} steps, {len(_log_chunk_start)} chunks")

if IS_EEF_MODE and _log_eef_target:
    final_eef = compute_eef_from_joints(q_after[RIGHT_SLICE], 1.0)
    init_eef = compute_eef_from_joints(ep_start_r, 1.0)
    eef_moved = float(np.linalg.norm(final_eef[:3] - init_eef[:3]))
    print(f"[real-replay] EEF pos moved: {eef_moved:.4f} m")
    print(f"[real-replay] final EEF: pos={np.round(final_eef[:3], 4)} euler={np.round(final_eef[3:6], 4)}")

try:
    robot.cancel_control()
except Exception:
    pass
if hasattr(robot, "disconnect"):
    robot.disconnect()

# [Initial Pose] (episode 후 복귀용)
위의 Step 2 코드와 동일합니다. Episode 완료 후 초기 자세로 복귀할 때 사용합니다.

In [ ]:
# ---------- [Reset] Initial Pose 복귀 ----------
# Step 2와 동일한 로직. Episode 후 다시 초기 자세로 이동.
if ENABLE_INITIAL_POSE:
    TORSO_INIT_DEG = np.array([0.0, 20.0, -40.0, 35.0, 0.0, 0.0], dtype=np.float64)
    INIT_RIGHT_DEG = np.array([-24.0, -60.0, 10.0, -120.0, -60.0, 85.0, 0.0], dtype=np.float64)
    INIT_LEFT_DEG = np.array([-24.0, 60.0, -10.0, -120.0, 60.0, 85.0, 0.0], dtype=np.float64)
    TORSO_INIT = np.deg2rad(TORSO_INIT_DEG)
    INIT_RIGHT = np.deg2rad(INIT_RIGHT_DEG)
    INIT_LEFT = np.deg2rad(INIT_LEFT_DEG)

    MID2_RIGHT = np.deg2rad(np.array([0.0, -15.0, 0.0, -100.0, 0.0, -70.0, 0.0], dtype=np.float64))
    MID2_LEFT = np.deg2rad(np.array([0.0, 15.0, 0.0, -100.0, 0.0, -70.0, 0.0], dtype=np.float64))

    robot_reset = rby.create_robot(ROBOT_IP, "a") if hasattr(rby, "create_robot") else rby.create_robot_a(ROBOT_IP)
    robot_reset.connect()
    assert robot_reset.is_connected()
    robot_reset.power_on(".*")
    robot_reset.servo_on(".*")
    robot_reset.enable_control_manager()
    try:
        robot_reset.cancel_control()
    except Exception:
        pass
    robot_reset.wait_for_control_ready(1000)

    head_target = HEAD_INIT if ENABLE_HEAD_INIT else None
    stream_reset = robot_reset.create_command_stream(priority=INIT_COMMAND_PRIORITY)

    waypoints = [
        ("midpoint2", None,       MID2_RIGHT, MID2_LEFT, 5.0),
        ("initial",   TORSO_INIT, INIT_RIGHT, INIT_LEFT, 3.0),
    ]
    for name, torso_target, right_target, left_target, duration in waypoints:
        print(f"[reset] move -> {name} (t={duration:.1f}s)")
        stream_reset = _move_waypoint_stream(
            robot_reset, stream_reset, name=name,
            torso_target=torso_target, right_target=right_target,
            left_target=left_target, head_target=head_target, duration=duration,
        )

    time.sleep(0.2)
    q_end = np.asarray(robot_reset.get_state().position, dtype=np.float64)
    print(f"[reset] done | right err={np.linalg.norm(q_end[RIGHT_SLICE]-INIT_RIGHT):.6f}")

    if USE_REMOTE_GRIPPER:
        for _arm in ("right", "left"):
            robot_reset.set_tool_flange_output_voltage(_arm, 12)
        time.sleep(0.5)

    try:
        robot_reset.cancel_control()
    except Exception:
        pass
    if hasattr(robot_reset, "disconnect"):
        robot_reset.disconnect()
    INITIAL_POSE_DONE = True
else:
    print("[reset] ENABLE_INITIAL_POSE=False -> skip")

# [Gripper Re-init] (episode 후 재초기화용)
위의 Step 2-5 코드와 동일합니다. Episode 완료 후 그리퍼를 다시 homing/열기할 때 사용합니다.

In [ ]:
# ---------- [Reset] Gripper 재초기화 ----------
if not globals().get("USE_REMOTE_GRIPPER", False):
    print("[gripper-reinit] USE_REMOTE_GRIPPER=False -> skip")
else:
    from rby1_remote_gripper import Gripper as RemoteGripper

    if "gripper_obj" in globals() and gripper_obj is not None:
        try:
            gripper_obj.stop()
        except Exception:
            pass
        gripper_obj = None

    gripper_obj = RemoteGripper()
    ok_init = gripper_obj.initialize(verbose=True)
    if not ok_init:
        raise RuntimeError("[gripper-reinit] 서버 응답 없음")

    ok_homing = gripper_obj.homing()
    if not ok_homing:
        raise RuntimeError("[gripper-reinit] homing 실패")
    print(f"[gripper-reinit] homing 완료 | min_q={gripper_obj.min_q} max_q={gripper_obj.max_q}")

    gripper_obj.start()
    gripper_obj.timeout = 3.0
    try:
        gripper_obj._sock.settimeout(3.0)
    except Exception:
        pass

    gripper_obj.set_normalized_target(np.array([1.0, 1.0]), wait_for_reply=True)
    time.sleep(0.5)
    GRIPPER_INIT_DONE = True
    print("[gripper-reinit] 완료 (양쪽 열림)")

---
## Action 로그 플롯 — 궤적 및 Chunk 경계 분석

Episode loop 실행 후 `_log_*` 버퍼를 기반으로 궤적을 시각화합니다.

| 모드 | 그래프 | 내용 |
|------|--------|------|
| Joint | 전체 궤적 | 관절별 commanded vs actual (deg) |
| Joint | Step delta | chunk 경계 spike 확인 |
| EEF | EEF 궤적 | target EEF [x,y,z,r,p,y] 포즈 추이 |
| EEF | Position delta | chunk 경계 spike 확인 |

In [ ]:
# ---------- Action chunk 로그 시각화 ----------
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

if "_log_cmd_right" not in globals() or len(_log_cmd_right) == 0:
    print("[plot] 로그 데이터 없음 — episode loop를 먼저 실행하세요.")
elif IS_EEF_MODE and "_log_eef_target" in globals() and len(_log_eef_target) > 0:
    # ============================================================
    # EEF 모드 플롯
    # ============================================================
    eef_targets = np.array(_log_eef_target)  # (N, 7)
    grip_r      = np.array(_log_cmd_grip)
    c_starts    = np.array(_log_chunk_start, dtype=int)
    N = eef_targets.shape[0]
    steps_x = np.arange(N)

    print(f"[plot] EEF mode | steps: {N}  |  chunks: {len(c_starts)}")

    # (A) EEF 궤적 (x,y,z,roll,pitch,yaw + gripper)
    eef_labels = ["x (m)", "y (m)", "z (m)", "roll (rad)", "pitch (rad)", "yaw (rad)", "gripper"]
    fig, axes = plt.subplots(2, 4, figsize=(20, 8), sharex=True)
    fig.suptitle("EEF Target Trajectory — LAP (EEF mode)", fontsize=13, fontweight="bold")
    axes_flat = axes.flatten()
    for j in range(7):
        ax = axes_flat[j]
        ax.plot(steps_x, eef_targets[:, j], color="steelblue", lw=1.2)
        for cs in c_starts:
            ax.axvline(x=cs, color="dimgray", linestyle="--", lw=0.8, alpha=0.7)
        ax.set_title(eef_labels[j], fontsize=10)
        ax.tick_params(labelsize=7)
        ax.grid(True, alpha=0.3)
    # 8th subplot: gripper
    ax_g = axes_flat[7]
    ax_g.plot(steps_x, grip_r, color="darkorange", lw=1.2)
    for cs in c_starts:
        ax_g.axvline(x=cs, color="dimgray", linestyle="--", lw=0.8, alpha=0.7)
    ax_g.set_title("R_Gripper (0=close, 1=open)", fontsize=10)
    ax_g.tick_params(labelsize=7)
    ax_g.grid(True, alpha=0.3)
    for ax in axes[1]:
        ax.set_xlabel("step", fontsize=8)
    fig.legend(
        handles=[plt.Line2D([0], [0], color="dimgray", linestyle="--", label="chunk start")],
        loc="lower center", ncol=1, fontsize=9, bbox_to_anchor=(0.5, -0.01),
    )
    plt.tight_layout(rect=[0, 0.04, 1, 1])
    plt.show()

    # (B) Step-to-step position delta norm
    pos_delta = np.linalg.norm(np.diff(eef_targets[:, :3], axis=0), axis=1)
    fig2, ax2 = plt.subplots(figsize=(16, 4))
    fig2.suptitle("Step-to-step EEF Position Delta (L2 norm, m)", fontsize=11, fontweight="bold")
    ax2.plot(np.arange(N - 1), pos_delta, color="steelblue", lw=0.9, label="pos delta")
    for k, cs in enumerate(c_starts[1:]):
        ax2.axvline(x=cs - 1, color="red", linestyle="--", lw=1.5, alpha=0.85,
                    label="chunk boundary" if k == 0 else None)
    ax2.set_ylabel("delta norm (m)", fontsize=9)
    ax2.set_xlabel("step", fontsize=9)
    ax2.grid(True, alpha=0.3)
    ax2.legend(fontsize=8)
    plt.tight_layout()
    plt.show()

    print("\n[plot] === EEF position delta stats ===")
    in_mask = np.ones(len(pos_delta), dtype=bool)
    for cs in c_starts[1:]:
        if 0 < cs - 1 < len(in_mask):
            in_mask[cs - 1] = False
    b_delta = pos_delta[~in_mask]
    i_delta = pos_delta[in_mask]
    print(
        f"  in-chunk avg={i_delta.mean():.5f} m | "
        f"boundary avg={b_delta.mean() if len(b_delta) else 0:.5f} m | "
        f"boundary max={b_delta.max() if len(b_delta) else 0:.5f} m"
    )

else:
    # ============================================================
    # Joint 모드 플롯 (기존 로직)
    # ============================================================
    cmd_r    = np.array(_log_cmd_right)
    grip_r   = np.array(_log_cmd_grip)
    act_r    = np.array(_log_actual_right) if _log_actual_right else np.empty((0, 7))
    act_x    = np.array(_log_actual_step,  dtype=int)
    c_starts = np.array(_log_chunk_start,  dtype=int)

    N = cmd_r.shape[0]
    steps_x = np.arange(N)

    print(f"[plot] Joint mode | steps: {N}  |  chunks: {len(c_starts)}  |  actual samples: {len(act_x)}")

    # (A) Right Arm 전체 궤적 (7 joints + gripper)
    fig, axes = plt.subplots(2, 4, figsize=(20, 8), sharex=True)
    fig.suptitle("Right Arm Trajectory (deg) — LAP", fontsize=13, fontweight="bold")
    axes_flat = axes.flatten()
    for j in range(7):
        ax = axes_flat[j]
        ax.plot(steps_x, np.degrees(cmd_r[:, j]), color="steelblue", lw=1.2, label="commanded")
        if len(act_x) > 0:
            ax.scatter(act_x, np.degrees(act_r[:, j]), color="tomato", s=18, zorder=5, label="actual")
        for cs in c_starts:
            ax.axvline(x=cs, color="dimgray", linestyle="--", lw=0.8, alpha=0.7)
        ax.set_title(f"R_J{j}", fontsize=10)
        ax.set_ylabel("deg", fontsize=8)
        ax.tick_params(labelsize=7)
        ax.grid(True, alpha=0.3)
    ax_g = axes_flat[7]
    ax_g.plot(steps_x, grip_r, color="darkorange", lw=1.2)
    for cs in c_starts:
        ax_g.axvline(x=cs, color="dimgray", linestyle="--", lw=0.8, alpha=0.7)
    ax_g.set_title("R_Gripper (0=close, 1=open)", fontsize=10)
    ax_g.set_ylabel("value", fontsize=8)
    ax_g.tick_params(labelsize=7)
    ax_g.grid(True, alpha=0.3)
    for ax in axes[1]:
        ax.set_xlabel("step", fontsize=8)
    handles = [
        mpatches.Patch(color="steelblue", label="commanded"),
        mpatches.Patch(color="tomato",    label="actual (sampled)"),
        plt.Line2D([0], [0], color="dimgray", linestyle="--", label="chunk start"),
    ]
    fig.legend(handles=handles, loc="lower center", ncol=3, fontsize=9, bbox_to_anchor=(0.5, -0.01))
    plt.tight_layout(rect=[0, 0.04, 1, 1])
    plt.show()

    # (B) Step-to-step delta norm
    delta_r = np.linalg.norm(np.diff(cmd_r, axis=0), axis=1)

    fig2, ax2 = plt.subplots(figsize=(16, 4))
    fig2.suptitle("Step-to-step Command Delta (L2 norm, rad) — Right Arm", fontsize=11, fontweight="bold")
    ax2.plot(np.arange(N - 1), delta_r, color="steelblue", lw=0.9, label="right arm delta")
    for k, cs in enumerate(c_starts[1:]):
        ax2.axvline(x=cs - 1, color="red", linestyle="--", lw=1.5, alpha=0.85,
                    label="chunk boundary" if k == 0 else None)
    ax2.set_ylabel("delta norm (rad)", fontsize=9)
    ax2.set_xlabel("step", fontsize=9)
    ax2.grid(True, alpha=0.3)
    ax2.legend(fontsize=8)
    plt.tight_layout()
    plt.show()

    print("\n[plot] === Step delta stats (chunk boundary vs internal) ===")
    in_mask = np.ones(len(delta_r), dtype=bool)
    for cs in c_starts[1:]:
        if 0 < cs - 1 < len(in_mask):
            in_mask[cs - 1] = False
    b_delta = delta_r[~in_mask]
    i_delta = delta_r[in_mask]
    print(
        f"  Right: in-chunk avg={i_delta.mean():.4f} rad | "
        f"boundary avg={b_delta.mean() if len(b_delta) else 0:.4f} rad | "
        f"boundary max={b_delta.max()  if len(b_delta) else 0:.4f} rad"
    )

---
## Step 6 — 환경 및 정리

에피소드 종료 후 카메라, 로봇, 그리퍼 리소스를 안전하게 정리합니다.

In [ ]:
# ---------- Step 6: 정리 ----------
# 카메라 파이프라인 정리
if "_rs_pipelines" in globals():
    for _name, _p in _rs_pipelines.items():
        try:
            _p.stop()
            print(f"[cleanup] camera {_name} stopped")
        except Exception as _e:
            print(f"[cleanup] camera {_name} stop 실패: {_e}")
    _rs_pipelines = {}

# 로봇 연결 정리
if "robot_obs" in globals() and robot_obs is not None:
    try:
        if hasattr(robot_obs, "disconnect"):
            robot_obs.disconnect()
        print("[cleanup] robot_obs disconnected")
    except Exception as _e:
        print(f"[cleanup] robot_obs disconnect 실패: {_e}")
    robot_obs = None

# 그리퍼 정리
if "gripper_obj" in globals() and gripper_obj is not None:
    try:
        gripper_obj.stop()
        print("[cleanup] gripper stopped")
    except Exception as _e:
        print(f"[cleanup] gripper stop 실패: {_e}")
    gripper_obj = None

ENV_SETUP_DONE = False
INITIAL_POSE_DONE = False
GRIPPER_INIT_DONE = False
print("[cleanup] done")